# Praktikum 6
ANN pada Dataset Nyata (Spotify Songs)

Lakukan percobaan penggunaan **ANNOY**, **FAISS**, dan **HNSWLIB** pada dataset sekunder berukuran besar (Micro Spotify). Pilih hanya fitur numerik, lakukan normalisasi menggunakan `StandardScaler`, lalu lakukan pencarian track terdekat dan bandingkan hasilnya.

> **Catatan:** `spotify_songs.csv` (32.833 lagu) berisi fitur audio seperti `danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, dan `tempo`.

> **Catatan teknis:** pada mesin lokal, menjalankan `sklearn` (exact NN), `faiss`, dan `hnswlib` dalam satu kernel dapat menyebabkan *deadlock* OpenMP. Untuk itu di awal notebook kita set `OMP_NUM_THREADS=1`. Ini tidak mengubah logika percobaan.

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'  # hindari konflik OpenMP antara sklearn, faiss, dan hnswlib

import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Load dataset
# -------------------------------
df = pd.read_csv('spotify_songs.csv')  # letakkan file di folder yang sama

features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
X = df[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10  # jumlah nearest neighbors

print("Jumlah lagu:", X_scaled.shape[0], "| jumlah fitur:", X_scaled.shape[1])
df[features].head()

Jumlah lagu: 32833 | jumlah fitur: 9


,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,0.748,0.916,-2.634,0.0583,0.1020,0.000000,0.0653,0.518,122.036
1,0.726,0.815,-4.969,0.0373,0.0724,0.004210,0.3570,0.693,99.972
2,0.675,0.931,-3.432,0.0742,0.0794,0.000023,0.1100,0.613,124.008
3,0.718,0.930,-3.778,0.1020,0.0287,0.000009,0.2040,0.277,121.956
4,0.650,0.833,-4.672,0.0359,0.0803,0.000000,0.0833,0.725,123.976


In [2]:
# -------------------------------
# Exact Nearest Neighbor (brute-force)
# -------------------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

Exact NN done in 3.577 s


In [3]:
# -------------------------------
# Annoy
# -------------------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

Annoy done in 0.750 s


In [4]:
# -------------------------------
# HNSW
# -------------------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
p_hnsw.init_index(max_elements=X_scaled.shape[0], ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, dist_hnsw = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

HNSW done in 1.182 s


In [5]:
# -------------------------------
# FAISS IVF
# -------------------------------
start = time.time()
quantizer = faiss.IndexFlatL2(X_scaled.shape[1])
index_faiss = faiss.IndexIVFFlat(quantizer, X_scaled.shape[1], 100, faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10
dist_faiss, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

FAISS IVF done in 0.926 s


In [6]:
# -------------------------------
# Contoh tampilkan top-5 neighbors dari item pertama
# -------------------------------
print("\nTop-5 neighbors for first song:")
print(f"Exact NN: {idx_exact[0][:5]}")
print(f"Annoy:    {idx_annoy[0][:5]}")
print(f"HNSW:     {idx_hnsw[0][:5]}")
print(f"FAISS:    {idx_faiss[0][:5]}")


Top-5 neighbors for first song:
Exact NN: [    0 29684 29734  4959 29713]
Annoy:    [0, 29684, 29734, 4959, 29713]
HNSW:     [    0 29684 29734  4959 29713]
FAISS:    [    0 29684 29734  4959 29713]


### Perbandingan Waktu dan Recall

In [7]:
def recall_at_k(pred, gt, k):
    correct = sum(len(set(pred[i][:k]) & set(gt[i][:k])) for i in range(len(pred)))
    return correct / (len(pred) * k)

tabel_spotify = pd.DataFrame([
    {'Metode': 'Exact NN', 'Waktu (s)': round(time_exact, 3), 'Recall@10': 1.0},
    {'Metode': 'Annoy', 'Waktu (s)': round(time_annoy, 3), 'Recall@10': round(recall_at_k(idx_annoy, idx_exact, k), 4)},
    {'Metode': 'HNSW', 'Waktu (s)': round(time_hnsw, 3), 'Recall@10': round(recall_at_k(idx_hnsw, idx_exact, k), 4)},
    {'Metode': 'FAISS IVF', 'Waktu (s)': round(time_faiss, 3), 'Recall@10': round(recall_at_k(idx_faiss, idx_exact, k), 4)},
])
tabel_spotify

,Metode,Waktu (s),Recall@10
0,Exact NN,3.577,1.0000
1,Annoy,0.750,0.8772
2,HNSW,1.182,0.9881
3,FAISS IVF,0.926,0.9924


### Analisis
**Alur code:**

1. **Memuat & menyiapkan data.** Dataset `spotify_songs.csv` dibaca, lalu 9 fitur audio numerik (`danceability` … `tempo`) dipilih. Karena skala tiap fitur berbeda jauh (misalnya `tempo` ratusan, `danceability` 0-1), dilakukan **StandardScaler** agar setiap fitur berkontribusi setara pada perhitungan jarak Euclidean. Tanpa normalisasi, `tempo` dan `loudness` akan mendominasi jarak.
2. **Exact NN (baseline).** `NearestNeighbors(algorithm='brute')` menghitung jarak ke **seluruh** 32.833 lagu untuk setiap query. Hasilnya 100% akurat dan menjadi *ground truth* untuk mengukur recall.
3. **Annoy.** Membangun 10 pohon proyeksi acak, lalu mencari tetangga tiap lagu. Cepat, tetapi karena bersifat *approximate*, hasilnya bisa sedikit berbeda dari exact.
4. **HNSW.** Membangun graf hierarkis; dengan `ef=200` hasilnya sangat mendekati exact dan tetap cepat.
5. **FAISS IVF.** Membagi ruang vektor menjadi `nlist=100` klaster (k-means) dan hanya mencari pada `nprobe=10` klaster terdekat, sehingga lebih cepat dari brute force.

**Kesimpulan:**

* Secara **waktu**, metode ANN (Annoy, HNSW, FAISS IVF) lebih cepat dibanding exact brute-force untuk dataset besar, terutama ketika jumlah data bertambah.
* Secara **akurasi (recall)**, pada percobaan ini HNSW (`ef=200`) dan FAISS IVF (`nprobe=10`) sama-sama mendekati exact (recall > 0,98), sedangkan Annoy dengan parameter default memberi recall lebih rendah (~0,88). Recall Annoy dapat dinaikkan dengan memperbesar `n_trees`/`search_k`.
* Untuk sistem rekomendasi lagu, trade-off ini penting: kita rela kehilangan sedikit akurasi demi latensi rendah saat melayani jutaan pengguna.
* Perhatikan bahwa nilai waktu sangat bergantung pada perangkat, jumlah thread, dan beban sistem, sehingga angka absolut bisa berbeda antar mesin.